<a href="https://colab.research.google.com/github/Namachivayan/semantic-search-system/blob/main/project1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install datasets

In [2]:
from datasets import load_dataset

# Load dataset
dataset = load_dataset("fancyzhx/ag_news")

print(dataset)

# Convert to pandas DataFrame
df = dataset["train"].to_pandas()

# Save as CSV
df.to_csv("ag_news_train.csv", index=False)

# Display first 5 rows
print(df.head())
df.columns
len(df)

README.md:   0%|          | 0.00/8.07k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 18.6MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.23MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})
                                                text  label
0  Wall St. Bears Claw Back Into the Black (Reute...      2
1  Carlyle Looks Toward Commercial Aerospace (Reu...      2
2  Oil and Economy Cloud Stocks' Outlook (Reuters...      2
3  Iraq Halts Oil Exports from Main Southern Pipe...      2
4  Oil prices soar to all-time record, posing new...      2


120000

In [3]:
label_map = {
    0: "World",
    1: "Sports",
    2: "Business",
    3: "Science & Technology"
}
df["document_id"] = range(1, len(df) + 1)
df["title"] = df["text"].str.split(".").str[0]
df[["title"]].head()
df["content"] = df["text"]
df[["content"]].head()
df["keywords"] = ""
df[["keywords"]].head()
df["category"] = df["label"].map(label_map) # Add this line to create the 'category' column
final_df = df[
    [
        "document_id",
        "category",
        "title",
        "content",
        "keywords"
    ]
]
final_df.head()

,document_id,category,title,content,keywords
0,1,Business,Wall St,Wall St. Bears Claw Back Into the Black (Reute...,
1,2,Business,Carlyle Looks Toward Commercial Aerospace (Reu...,Carlyle Looks Toward Commercial Aerospace (Reu...,
2,3,Business,Oil and Economy Cloud Stocks' Outlook (Reuters...,Oil and Economy Cloud Stocks' Outlook (Reuters...,
3,4,Business,Iraq Halts Oil Exports from Main Southern Pipe...,Iraq Halts Oil Exports from Main Southern Pipe...,
4,5,Business,"Oil prices soar to all-time record, posing new...","Oil prices soar to all-time record, posing new...",


In [4]:
final_df.to_csv("semantic_search_dataset.csv", index=False)

In [5]:
print(final_df.info())
print(final_df.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 120000 entries, 0 to 119999
Data columns (total 5 columns):
 #   Column       Non-Null Count   Dtype 
---  ------       --------------   ----- 
 0   document_id  120000 non-null  int64 
 1   category     120000 non-null  object
 2   title        120000 non-null  object
 3   content      120000 non-null  object
 4   keywords     120000 non-null  object
dtypes: int64(1), object(4)
memory usage: 4.6+ MB
None
   document_id  category                                              title  \
0            1  Business                                            Wall St   
1            2  Business  Carlyle Looks Toward Commercial Aerospace (Reu...   
2            3  Business  Oil and Economy Cloud Stocks' Outlook (Reuters...   
3            4  Business  Iraq Halts Oil Exports from Main Southern Pipe...   
4            5  Business  Oil prices soar to all-time record, posing new...   

                                             content keywords  
0 

In [6]:
import re
def clean_text(text):
    # Convert to lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(r"http\S+|www\S+", "", text)

    # Remove HTML tags
    text = re.sub(r"<.*?>", "", text)

    # Keep only letters, numbers and spaces
    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text



In [7]:
final_df["content"] = final_df["content"].apply(clean_text)
final_df.head()

/tmp/ipykernel_689/1440747331.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_df["content"] = final_df["content"].apply(clean_text)


,document_id,category,title,content,keywords
0,1,Business,Wall St,wall st bears claw back into the black reuters...,
1,2,Business,Carlyle Looks Toward Commercial Aerospace (Reu...,carlyle looks toward commercial aerospace reut...,
2,3,Business,Oil and Economy Cloud Stocks' Outlook (Reuters...,oil and economy cloud stocks outlook reuters r...,
3,4,Business,Iraq Halts Oil Exports from Main Southern Pipe...,iraq halts oil exports from main southern pipe...,
4,5,Business,"Oil prices soar to all-time record, posing new...",oil prices soar to all time record posing new ...,


In [8]:
!pip install keybert sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.4/41.4 kB 2.2 MB/s eta 0:00:00


In [9]:
from keybert import KeyBERT
kw_model = KeyBERT()

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [10]:
text = final_df.loc[0, "content"]

keywords = kw_model.extract_keywords(
    text,
    keyphrase_ngram_range=(1,2),
    stop_words="english",
    top_n=5
)

print(keywords)


[('wall st', 0.5405), ('st bears', 0.5025), ('bears claw', 0.4437), ('wall street', 0.4275), ('sellers wall', 0.4273)]


In [11]:
keyword_list = [k[0] for k in keywords]

print(keyword_list)

['wall st', 'st bears', 'bears claw', 'wall street', 'sellers wall']


In [12]:
final_df.loc[0, "keywords"] = ", ".join(keyword_list)
final_df.head()

,document_id,category,title,content,keywords
0,1,Business,Wall St,wall st bears claw back into the black reuters...,"wall st, st bears, bears claw, wall street, se..."
1,2,Business,Carlyle Looks Toward Commercial Aerospace (Reu...,carlyle looks toward commercial aerospace reut...,
2,3,Business,Oil and Economy Cloud Stocks' Outlook (Reuters...,oil and economy cloud stocks outlook reuters r...,
3,4,Business,Iraq Halts Oil Exports from Main Southern Pipe...,iraq halts oil exports from main southern pipe...,
4,5,Business,"Oil prices soar to all-time record, posing new...",oil prices soar to all time record posing new ...,


In [13]:
for i in range(100):
    text = final_df.loc[i, "content"]

    keywords = kw_model.extract_keywords(
        text,
        keyphrase_ngram_range=(1,2),
        stop_words="english",
        top_n=5
    )

    keyword_list = [k[0] for k in keywords]

    final_df.loc[i, "keywords"] = ", ".join(keyword_list)

In [14]:
!pip install sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 50.6 MB/s eta 0:00:00


In [15]:
from sentence_transformers import SentenceTransformer

In [16]:
model = SentenceTransformer("all-MiniLM-L6-v2")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [17]:
print(df.columns)

Index(['text', 'label', 'document_id', 'title', 'content', 'keywords',
       'category'],
      dtype='object')


In [18]:
embedding = model.encode(df.loc[0, "content"])

print(embedding)

[ 7.43859494e-03  2.85623856e-02  4.10954878e-02  1.05001405e-01
  2.32820306e-02  3.51258591e-02 -2.14059930e-02 -2.29023080e-02
  4.94522322e-03 -6.68985620e-02 -5.93040325e-02  2.45948173e-02
 -5.12895510e-02 -4.04271595e-02  6.65567408e-04 -1.17221139e-02
 -4.34821891e-03  2.06902418e-02 -2.29348894e-03 -3.47106196e-02
 -8.01126659e-02 -6.02049828e-02 -2.86629219e-02 -1.86906159e-02
 -7.84591362e-02  4.60197292e-02 -4.96721193e-02 -3.33420373e-02
  5.32347448e-02 -1.26492038e-01 -7.07766935e-02 -6.30829774e-04
 -2.26125419e-02  2.00371374e-03  8.22302978e-03 -6.30246615e-03
  8.53442121e-03 -4.64320779e-02  4.35816124e-02  3.90969478e-02
  1.36761914e-03 -2.62946226e-02 -5.70246726e-02  3.29648294e-02
  1.93821136e-02  4.52637151e-02  5.92519119e-02  5.84027357e-02
 -2.66226032e-03 -3.10069490e-02  1.38102081e-02 -2.49957554e-02
 -3.08281183e-02  3.99599448e-02  2.42338702e-02  1.04000501e-03
  4.13026698e-02 -3.98771018e-02  1.06792070e-01 -5.37586324e-02
  8.35924968e-02  2.60991

In [19]:
print(embedding.shape)

(384,)


In [20]:
embeddings = model.encode(
    df["content"].tolist(),
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True
)

Batches:   0%|          | 0/3750 [00:00<?, ?it/s]

In [21]:
import numpy as np

np.save("document_embeddings.npy", embeddings)

In [22]:
!pip install -q faiss-cpu

In [23]:
import faiss
import numpy as np

In [24]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

print("FAISS index created")

FAISS index created


In [25]:
index.add(embeddings)
print("Total vectors:", index.ntotal)

Total vectors: 120000


In [43]:
def semantic_search(query, top_k=5, threshold=0.45):

    query = query.strip()

    if not query:
        return []

    # Query embedding
    query_embedding = model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    # FAISS search
    scores, indices = index.search(
        query_embedding,
        top_k
    )

    results = []

    for score, idx in zip(scores[0], indices[0]):

        # Important relevance filter
        if float(score) < threshold:
            continue

        row = final_df.iloc[int(idx)]

        results.append({
            "document_id": int(row["document_id"]),
            "category": row["category"],
            "title": row["title"],
            "content": row["content"],
            "keywords": row["keywords"],
            "score": float(score)
        })

    return results

In [44]:
semantic_search("artificial inteligence")

[{'document_id': 742,
  'category': 'Business',
  'title': "Computers with multiple personalities The jury's still out on whether a computer can ever truly be intelligent, but there's no question that it can have multiple personalities",
  'content': 'computers with multiple personalities the jury s still out on whether a computer can ever truly be intelligent but there s no question that it can have multiple personalities it s just a matter of software',
  'keywords': '',
  'score': 0.4722561836242676},
 {'document_id': 554,
  'category': 'Science & Technology',
  'title': 'NASA Develops Robust Artificial Intelligence for Planetary Rovers NASA is planning to add a strong dose of artificial intelligence (AI) to planetary rovers to make them much more self-reliant, capable of making basic decisions during a mission',
  'content': 'nasa develops robust artificial intelligence for planetary rovers nasa is planning to add a strong dose of artificial intelligence ai to planetary rovers to m

In [45]:
threshold=0.45

In [46]:
results = semantic_search(
    "artificial intelligence",
    top_k=5,
    threshold=0.45
)

for r in results:
    print("Category:", r["category"])
    print("Title:", r["title"])
    print("Score:", r["score"])
    print("-" * 60)

Category: Science & Technology
Title: NASA Develops Robust Artificial Intelligence for Planetary Rovers NASA is planning to add a strong dose of artificial intelligence (AI) to planetary rovers to make them much more self-reliant, capable of making basic decisions during a mission
Score: 0.5102806687355042
------------------------------------------------------------
Category: Science & Technology
Title: NASA Software Allows Satellites to Troubleshoot in Space Goddard Space Flight Center -- NASA scientists recently successfully radioed artificial intelligence (AI) software to a satellite
Score: 0.4800454378128052
------------------------------------------------------------


In [47]:
results = semantic_search(
    "xyzabc123 moon pizza helicopter",
    top_k=5,
    threshold=0.45
)

print("Number of results:", len(results))

Number of results: 0


In [ ]:
import gradio as gr


def search_articles(query):

    if not query.strip():
        return "⚠️ Please enter a search query."

    results = semantic_search(
        query,
        top_k=5,
        threshold=0.45
    )

    if len(results) == 0:
        return """
## ❌ No Relevant Results

I couldn't find a sufficiently relevant article
for your search.

Try searching about:
- 🏏 Cricket / Sports
- 💼 Business / Stock Market
- 🌍 World News
- 💻 Technology / AI
"""

    output = f"## 🔎 Search Results for: `{query}`\n\n"

    for i, r in enumerate(results):

        output += f"""
### {i+1}. {r["title"]}

**Category:** `{r["category"]}`
**Similarity Score:** `{r["score"]:.3f}`

**Content:**
{r["content"]}

---

"""

    return output


with gr.Blocks(
    title="Semantic Search System"
) as demo:

    gr.Markdown("""
# 🔎 Semantic Search System

### Search news articles based on semantic meaning

Enter a query below and find the most relevant articles.
""")

    query = gr.Textbox(
        label="Search Query",
        placeholder="Example: artificial intelligence",
        lines=2
    )

    search_btn = gr.Button(
        "🔍 Search",
        variant="primary"
    )

    output = gr.Markdown()

    search_btn.click(
        fn=search_articles,
        inputs=query,
        outputs=output
    )

    query.submit(
        fn=search_articles,
        inputs=query,
        outputs=output
    )


demo.launch(
    share=True,
    debug=True
)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://d8fc9bf5e8f8982672.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
